# 06 · Building and Testing Each Agent in Isolation

Before wiring agents together (notebook 07), each one is built and tested **on its own, with fixed inputs**.
This is the agent equivalent of unit testing: if the full pipeline produces a bad report, we want to already know
each component behaves correctly.

| Agent | Input | Output | Built in |
|---|---|---|---|
| Planner | user query | `ResearchPlan` | notebook 02 ✔ |
| Researcher | one sub-question | `Finding` | notebook 04 ✔ |
| **Fact-checker** | all findings + their sources | `FactCheckReport` | here |
| **Writer** | plan + verified claims + numbered sources | Markdown report with `[n]` citations | here |
| **Critic** | plan + report | `Critique` → approve / follow-up questions | here |

### Design principle: deterministic code where possible, LLM only where judgement is needed
LLM calls are slow, cost quota and can be wrong. So each agent combines:
- **deterministic Python** for what rules can decide (URL grounding, source credibility, citation numbering,
  approval thresholds), and
- **an LLM** only for genuine judgement (do two claims contradict? is this report complete?).

### Test fixtures
We use **hand-written, synthetic findings** (clearly fake numbers and domains) with **planted problems**:
a contradiction between two researchers and a claim citing a URL that was never retrieved. A good fact-checker
must catch both. Fixed fixtures make the test repeatable and the results comparable across prompt versions.

In [1]:
from nb_setup import setup, timer
ROOT = setup()

from nb_components import structured, llm_for, ResearchPlan, SubQuestion
from typing import Literal
from pydantic import BaseModel, Field
from urllib.parse import urlparse
from datetime import date

API keys:
  GROQ_API_KEY         MISSING
  OPENROUTER_API_KEY   set
  TAVILY_API_KEY       set
  LANGSMITH_API_KEY    set
  LangSmith tracing    true


LLM cache: data\llm_cache.sqlite


## 1 · Schemas and fixtures

The `Finding` from notebook 04 is extended with `retrieved_urls`: the URLs that actually appeared in the
researcher's tool results. The researcher *node* fills this deterministically from its `ToolMessage`s, so it is
ground truth, not something the LLM claims.

In [2]:
class Claim(BaseModel):
    statement: str
    source_urls: list[str] = Field(min_length=1)


class Finding(BaseModel):
    sub_question: str
    summary: str
    claims: list[Claim]
    confidence: Literal["high", "medium", "low"]
    gaps: list[str] = []
    retrieved_urls: list[str] = Field(default_factory=list, description="Filled by code from tool outputs")


PLAN = ResearchPlan(
    objective="Assess how AI coding agents are changing software engineering workflows in 2026",
    research_type="trend_analysis",
    sub_questions=[
        SubQuestion(question="How widely are AI coding agents adopted by software teams in 2026?",
                    rationale="Establish scale", search_strategy="web"),
        SubQuestion(question="What measurable productivity effects of AI coding agents have studies reported?",
                    rationale="Evidence of impact", search_strategy="academic"),
        SubQuestion(question="What risks and limitations of AI coding agents do teams report?",
                    rationale="Balance", search_strategy="web"),
    ],
    report_outline=["Executive summary", "Adoption", "Productivity evidence", "Risks", "Outlook"],
)

# SYNTHETIC FIXTURES: fake numbers and domains, with deliberately planted problems.
FINDINGS = [
    Finding(
        sub_question=PLAN.sub_questions[0].question,
        summary="Adoption is high: most surveyed developers use coding agents weekly.",
        claims=[
            Claim(statement="58% of surveyed teams use AI coding agents daily.",
                  source_urls=["https://devsurvey.example.org/2026"]),
            Claim(statement="Agent usage roughly doubled between 2025 and 2026.",
                  source_urls=["https://technews.example.com/agents-2026"]),
        ],
        confidence="high",
        retrieved_urls=["https://devsurvey.example.org/2026", "https://technews.example.com/agents-2026"],
    ),
    Finding(
        sub_question=PLAN.sub_questions[1].question,
        summary="Controlled studies report moderate productivity gains, concentrated in routine tasks.",
        claims=[
            Claim(statement="A controlled study found 21% faster completion of routine coding tasks.",
                  source_urls=["https://arxiv.org/abs/2601.00001"]),
            Claim(statement="Fewer than 30% of teams use coding agents on a daily basis.",       # PLANTED: contradicts 58%
                  source_urls=["https://arxiv.org/abs/2601.00002"]),
        ],
        confidence="medium",
        retrieved_urls=["https://arxiv.org/abs/2601.00001", "https://arxiv.org/abs/2601.00002"],
    ),
    Finding(
        sub_question=PLAN.sub_questions[2].question,
        summary="Teams report review burden and subtle bugs as main risks.",
        claims=[
            Claim(statement="Code review time increased for agent-generated pull requests.",
                  source_urls=["https://engblog.example.net/review-burden"]),
            Claim(statement="Agent-written code has 3x more security vulnerabilities.",          # PLANTED: URL never retrieved
                  source_urls=["https://made-up-source.example.io/vulns"]),
        ],
        confidence="medium",
        gaps=["No independent data on long-term maintenance cost"],
        retrieved_urls=["https://engblog.example.net/review-burden", "https://en.wikipedia.org/wiki/AI_agent"],
    ),
]
print(sum(len(f.claims) for f in FINDINGS), "claims across", len(FINDINGS), "findings")

6 claims across 3 findings


## 2 · Fact-checker

### 2a · Deterministic checks (free, instant, 100% reliable)
1. **Grounding**: every cited URL must be in the researcher's `retrieved_urls`. Otherwise the model invented or
   misattributed it → claim marked `unverified`.
2. **Source credibility**: a transparent rule-based score by domain type. Not perfect, but explainable, and the
   Writer can prefer high-credibility sources.

In [3]:
HIGH = ("arxiv.org", ".gov", ".edu", "nature.com", "acm.org", "ieee.org")
MEDIUM = ("wikipedia.org", "github.com", "reuters.com", "apnews.com")


def credibility(url: str) -> float:
    """Rule-based source credibility in [0, 1] by domain type."""
    host = urlparse(url).netloc.lower()
    if any(h in host for h in HIGH):
        return 0.9
    if any(m in host for m in MEDIUM):
        return 0.7
    return 0.5


def grounding_check(findings: list[Finding]) -> list[dict]:
    rows = []
    for f in findings:
        for c in f.claims:
            ungrounded = [u for u in c.source_urls if u not in f.retrieved_urls]
            rows.append({"statement": c.statement, "source_urls": c.source_urls,
                         "grounded": not ungrounded,
                         "credibility": max((credibility(u) for u in c.source_urls), default=0.0)})
    return rows


import pandas as pd
grounding = grounding_check(FINDINGS)
pd.DataFrame(grounding)[["statement", "grounded", "credibility"]]

,statement,grounded,credibility
0,58% of surveyed teams use AI coding agents daily.,True,0.5
1,Agent usage roughly doubled between 2025 and 2...,True,0.5
2,A controlled study found 21% faster completion...,True,0.9
3,Fewer than 30% of teams use coding agents on a...,True,0.9
4,Code review time increased for agent-generated...,True,0.5
5,Agent-written code has 3x more security vulner...,False,0.5


### 2b · LLM check: contradictions across researchers

Researchers work in parallel and **never see each other's results**, so contradictions between them are expected.
Detecting that "58% use agents daily" conflicts with "fewer than 30% daily" requires semantic understanding,
which is the LLM's job. We give it all *grounded* claims, numbered, and ask for structured conflicts.

In [4]:
class Conflict(BaseModel):
    """Two claims that cannot both be true as stated."""
    claim_ids: list[int] = Field(min_length=2, max_length=2, description="Ids of the two conflicting claims")
    explanation: str = Field(description="Why they conflict and what could explain the difference")


class ConflictReport(BaseModel):
    conflicts: list[Conflict] = Field(default_factory=list, description="Empty if no conflicts")


FACT_CHECK_PROMPT = """You are a rigorous fact-checker. Below are numbered claims gathered by different researchers.
Identify pairs of claims that CONTRADICT each other (they cannot both be true as stated).
Differences in scope or wording that can both be true are NOT conflicts.
Return an empty list if there are no real contradictions.

Claims:
{claims}"""

grounded_claims = [g for g in grounding if g["grounded"]]
numbered = "\n".join(f"[{i}] {g['statement']} (source: {g['source_urls'][0]})" for i, g in enumerate(grounded_claims))

with timer("fact-check LLM"):
    conflict_report = structured(ConflictReport, "reasoning").invoke(FACT_CHECK_PROMPT.format(claims=numbered))

for c in conflict_report.conflicts:
    a, b = (grounded_claims[i]["statement"] for i in c.claim_ids)
    print(f"CONFLICT:\n  - {a}\n  - {b}\n  why: {c.explanation}\n")

[fact-check LLM] 15.03s
CONFLICT:
  - 58% of surveyed teams use AI coding agents daily.
  - Fewer than 30% of teams use coding agents on a daily basis.
  why: Claim [0] states that 58% of surveyed teams use AI coding agents daily, while claim [3] states that fewer than 30% of teams use coding agents daily. These cannot both be true as stated because 58% is not fewer than 30%.



### 2c · Combine into the fact-check report
Each claim gets a status: **supported** (grounded, no conflict), **contested** (grounded but in a conflict), or
**unverified** (citation not grounded → excluded from the report). The Writer receives only supported + contested
claims, and must present contested ones *as* contested.

In [5]:
class VerifiedClaim(BaseModel):
    statement: str
    source_urls: list[str]
    status: Literal["supported", "contested", "unverified"]
    credibility: float


class FactCheckReport(BaseModel):
    claims: list[VerifiedClaim]
    conflicts: list[str]


def build_fact_check(grounding: list[dict], conflicts: ConflictReport) -> FactCheckReport:
    grounded = [g for g in grounding if g["grounded"]]
    contested = {grounded[i]["statement"] for c in conflicts.conflicts for i in c.claim_ids}
    claims = [VerifiedClaim(statement=g["statement"], source_urls=g["source_urls"], credibility=g["credibility"],
                            status="unverified" if not g["grounded"]
                            else "contested" if g["statement"] in contested else "supported")
              for g in grounding]
    return FactCheckReport(claims=claims, conflicts=[c.explanation for c in conflicts.conflicts])


fact_check = build_fact_check(grounding, conflict_report)
pd.DataFrame([c.model_dump() for c in fact_check.claims])[["statement", "status", "credibility"]]

,statement,status,credibility
0,58% of surveyed teams use AI coding agents daily.,contested,0.5
1,Agent usage roughly doubled between 2025 and 2...,supported,0.5
2,A controlled study found 21% faster completion...,supported,0.9
3,Fewer than 30% of teams use coding agents on a...,contested,0.9
4,Code review time increased for agent-generated...,supported,0.5
5,Agent-written code has 3x more security vulner...,unverified,0.5


In [6]:
# Did the fact-checker catch both planted problems?
status = {c.statement: c.status for c in fact_check.claims}
caught_contradiction = (status["58% of surveyed teams use AI coding agents daily."] == "contested"
                        and status["Fewer than 30% of teams use coding agents on a daily basis."] == "contested")
caught_fake_url = status["Agent-written code has 3x more security vulnerabilities."] == "unverified"
print(f"planted contradiction caught: {caught_contradiction}")
print(f"planted fake citation caught: {caught_fake_url}")

planted contradiction caught: True
planted fake citation caught: True


## 3 · Writer

Citation handling is where LLM report writers usually fail (wrong numbers, invented URLs). Our approach:
1. **Code** assigns each unique source a number `[1]..[n]` and gives the Writer a numbered source list.
2. The Writer only writes `[n]` markers, **never URLs**.
3. **Code** validates the markers (every `[n]` must exist) and appends the References section itself.

So the reference list is always correct by construction. The Writer uses the *worker* model at a slightly higher
temperature for more natural prose (notebook 01 decision).

In [7]:
import re


def number_sources(fc: FactCheckReport) -> dict[str, int]:
    """Assign [n] numbers to unique URLs of usable claims, in order of first appearance."""
    numbers: dict[str, int] = {}
    for c in fc.claims:
        if c.status != "unverified":
            for u in c.source_urls:
                numbers.setdefault(u, len(numbers) + 1)
    return numbers


SOURCE_NUMBERS = number_sources(fact_check)

evidence = "\n".join(
    f"- ({c.status}) {c.statement} " + " ".join(f"[{SOURCE_NUMBERS[u]}]" for u in c.source_urls)
    for c in fact_check.claims if c.status != "unverified")

WRITER_PROMPT = """You are an expert research writer. Today's date is {today}.
Write a well-structured Markdown research report.

Objective: {objective}
Use these section headings, in order: {outline}

Evidence (each line ends with its citation markers):
{evidence}

Known conflicts between sources:
{conflicts}

Rules:
- Use ONLY the evidence above. Do not add facts from memory.
- Cite with the given markers like [1] or [2][3] right after the sentence they support. Never write URLs.
- Claims marked (contested) must be presented as disputed, mentioning the disagreement.
- Start with a '# ' title. Be concise: about 300-450 words. Do NOT write a references section."""

writer_input = WRITER_PROMPT.format(
    today=date.today().isoformat(), objective=PLAN.objective, outline=" | ".join(PLAN.report_outline),
    evidence=evidence, conflicts="\n".join(f"- {c}" for c in fact_check.conflicts) or "None")

with timer("writer"):
    draft = llm_for("worker", temperature=0.3).invoke(writer_input).content
print(draft)

[writer] 13.94s
# AI Coding Agents in Software Engineering Workflows in 2026

## Executive summary
AI coding agents are reshaping software engineering workflows in 2026, but the evidence is mixed. Adoption appears to have grown, with agent usage roughly doubling between 2025 and 2026 [2]. However, estimates of daily use are disputed: one source reports that 58% of surveyed teams use AI coding agents daily [1], while another says fewer than 30% do so [4]. Productivity evidence is also limited: a controlled study found 21% faster completion of routine coding tasks [3], but code review time increased for agent-generated pull requests [5].

## Adoption
Adoption of AI coding agents increased substantially from 2025 to 2026, with usage roughly doubling over that period [2]. The extent of routine, daily adoption remains contested. One source reports that 58% of surveyed teams use AI coding agents daily [1]. Another source reports that fewer than 30% of teams use coding agents on a daily basis

In [8]:
def validate_and_finalize(markdown: str, numbers: dict[str, int]) -> tuple[str, list[str]]:
    """Check citation markers and append a references list built by code (never by the LLM)."""
    cited = {int(n) for n in re.findall(r"\[(\d+)\]", markdown)}
    valid = set(numbers.values())
    problems = [f"invalid citation [{n}]" for n in sorted(cited - valid)]
    problems += [f"source [{n}] never cited" for n in sorted(valid - cited)]
    refs = "\n".join(f"{n}. {url}" for url, n in sorted(numbers.items(), key=lambda kv: kv[1]))
    return f"{markdown.rstrip()}\n\n## References\n{refs}\n", problems


report, citation_problems = validate_and_finalize(draft, SOURCE_NUMBERS)
print("citation problems:", citation_problems or "none")
print(report[-400:])

citation problems: none
refore not simply faster development, but a reallocation of effort toward review and oversight. Daily adoption will remain uncertain until the conflicting estimates are resolved [1][4].

## References
1. https://devsurvey.example.org/2026
2. https://technews.example.com/agents-2026
3. https://arxiv.org/abs/2601.00001
4. https://arxiv.org/abs/2601.00002
5. https://engblog.example.net/review-burden



## 4 · Critic

The Critic grades the report against a rubric and, if it finds gaps, proposes **follow-up questions**, which become
new researcher tasks in the loop (notebook 05 §3).

Two design choices:
- **Scores, not a verdict.** LLM judges tend to be lenient when asked "is this good?". Asking for 1–5 scores per
  criterion with justifications is more discriminating.
- **Code decides approval** from the scores (threshold), so the policy is explicit and tunable, not hidden in a prompt.

In [9]:
class Critique(BaseModel):
    """Rubric-based evaluation of a research report."""
    coverage: int = Field(ge=1, le=5, description="Does the report answer every part of the objective?")
    evidence_use: int = Field(ge=1, le=5, description="Are claims supported by citations; contested points flagged?")
    coherence: int = Field(ge=1, le=5, description="Is it well structured, clear and non-repetitive?")
    justification: str = Field(description="2-3 sentences explaining the scores")
    gaps: list[str] = Field(default_factory=list, description="Specific missing information")
    follow_up_questions: list[str] = Field(default_factory=list, max_length=3,
                                           description="Up to 3 searchable questions that would fill the gaps")


CRITIC_PROMPT = """You are a demanding research editor. Evaluate the report against the objective.
Score strictly: 5 = excellent, 3 = acceptable with clear weaknesses, 1 = poor.
Only propose follow-up questions for gaps that new web/academic research could realistically fill.

Objective: {objective}
Planned sub-questions:
{sub_questions}

Report:
{report}"""

APPROVAL_THRESHOLD = 4   # every criterion must reach this


def is_approved(c: Critique) -> bool:
    return min(c.coverage, c.evidence_use, c.coherence) >= APPROVAL_THRESHOLD


with timer("critic"):
    critique = structured(Critique, "reasoning").invoke(CRITIC_PROMPT.format(
        objective=PLAN.objective, report=report,
        sub_questions="\n".join(f"- {s.question}" for s in PLAN.sub_questions)))

print(f"coverage={critique.coverage} evidence_use={critique.evidence_use} coherence={critique.coherence}"
      f" -> approved={is_approved(critique)}")
print("why:", critique.justification)
print("gaps:", critique.gaps)
print("follow-ups:", critique.follow_up_questions)

[critic] 13.29s
coverage=4 evidence_use=4 coherence=5 -> approved=True
why: The report addresses all three sub-questions and explicitly flags the conflicting daily-adoption estimates, but it does not resolve how widely agents are adopted or enumerate a broad range of team-reported risks beyond review burden. Claims are supported by in-text citations and contested points are clearly noted, though the cited URLs appear illustrative rather than verifiable. The structure is logical, concise, and easy to follow.
gaps: ['Methodology and context of the 21% productivity study (sample size, task types, programming languages, statistical significance)', 'A comprehensive list of risks and limitations reported by teams (e.g., security vulnerabilities, technical debt, skill atrophy, integration issues)', 'Adoption breakdown by team size, industry, or geographic region in 2026']
follow-ups: ['What risks and limitations of AI coding agents are most frequently reported by software engineering teams in

## 5 · Observations & Decisions

*Recorded from the run on 2026-11-19 (LLM outputs vary between runs).*

**Observations**
- **Both planted problems were caught.** Deterministic grounding flagged the fake citation instantly (no LLM); the
  reasoning model (~15 s) found exactly one conflict, the planted 58% vs <30% contradiction, with no false positives.
- The Writer (~14 s) presented the contested adoption figures *as disputed* and produced **zero citation problems**
  (no invalid or unused markers); References were generated by code, so they are correct by construction.
- The Critic (~13 s) scored coverage 4 / evidence 4 / coherence 5 → approved, yet still listed concrete gaps
  (study methodology, broader risk list). Useful as follow-ups even when approved.
- A full fact-check → write → critique pass costs **3 LLM calls (~42 s)** on free models.

**Decisions**
1. **Fact-checker** = deterministic grounding + credibility scoring, then one structured LLM call for conflicts.
   Ungrounded claims are excluded; contested claims are passed to the Writer flagged as contested.
2. **Writer** = worker model, T=0.3; receives numbered evidence; writes `[n]` markers only; code validates markers
   and builds References.
3. **Critic** = reasoning model, rubric (coverage, evidence_use, coherence 1–5) + ≤3 follow-up questions;
   approval = all scores ≥ 4 (configurable). Follow-ups become new `Send` researcher tasks.
4. Schemas `Claim`, `Finding` (+`retrieved_urls`), `VerifiedClaim`, `FactCheckReport`, `Critique` and the prompts move
   to `src/research_system/schemas/` and `prompts/`.